In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import make_regression
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split

from deepordinal.tf import OrdinalOutput, ordinal_loss

In [ ]:
x, y = make_regression(n_samples=10000, n_features=8, n_informative=6, random_state=0)
y = QuantileTransformer().fit_transform(y.reshape(-1, 1))[:, 0]
y = np.clip(np.floor(y * 4), 0, 3).astype(np.int32)  # 4 ordinal classes: 0, 1, 2, 3

x_train, x_val, y_train, y_val = train_test_split(
    x.astype(np.float32), y, test_size=0.25, random_state=0
)
print("Class distribution:", np.bincount(y_train))

In [ ]:
tf.keras.utils.set_random_seed(0)
features = tf.keras.layers.Dense(16, activation="relu")
head = OrdinalOutput(output_dim=4)  # head(features(x)) returns class probabilities
head(features(x_train[:1]))  # build both layers so their variables exist
params = features.trainable_variables + head.trainable_variables
optimizer = tf.keras.optimizers.Adam()

In [ ]:
def loss_fn(xb, yb):
    return ordinal_loss(head.logit(features(xb)), yb, head.interior_thresholds)


@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        loss = loss_fn(xb, yb)
    optimizer.apply_gradients(zip(tape.gradient(loss, params), params))
    return loss


train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(x_train), seed=0).batch(16)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(16)

epochs = 50
train_losses, val_losses = [], []
for epoch in range(epochs):
    train_losses.append(np.mean([train_step(xb, yb).numpy() for xb, yb in train_ds]))
    val_losses.append(np.mean([loss_fn(xb, yb).numpy() for xb, yb in val_ds]))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
preds = tf.argmax(head(features(x_val)), axis=1).numpy()
print(f"Validation accuracy: {np.mean(preds == y_val):.4f}")

In [ ]:
plt.plot(train_losses, "--", label="train_loss")
plt.plot(val_losses, label="val_loss")
plt.title("Training and validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()